# 🚛 German Logistics Network Optimizer (TSP)

**Objective:** Solving the Traveling Salesman Problem (TSP) for 10 major German logistics hubs using Combinatorial Optimization and Linear Programming.

**Academic Context:** This project bridges the abstract mathematical theory from my **Semester 10 "Optimization Techniques"** coursework with applied Python programming. By utilizing Graph Theory and the `Google OR-Tools` suite, this algorithm calculates the global minimum travel distance required to visit every node in the network and return to the depot.

In [5]:
# Import the required optimization library
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
print("Libraries imported successfully!")

Libraries imported successfully!


### 1. Defining the Network (The Distance Matrix)
We model the cities as nodes in a complete, weighted graph. The weights are the driving distances (in kilometers) between 10 major German economic centers. The starting depot is set to Berlin (Index 0).

In [6]:
# Distance matrix in kilometers between major German cities
distance_matrix = [
    [0, 585, 289, 545, 578, 633, 564, 190, 495, 532], # 0: Berlin
    [585, 0, 775, 393, 576, 232, 612, 430, 588, 630], # 1: Munich
    [289, 775, 0, 493, 428, 655, 400, 395, 344, 365], # 2: Hamburg
    [545, 393, 493, 0, 191, 205, 228, 393, 218, 253], # 3: Frankfurt
    [578, 576, 428, 191, 0, 370, 40, 471, 72, 35],    # 4: Cologne
    [633, 232, 655, 205, 370, 0, 400, 480, 412, 432], # 5: Stuttgart
    [564, 612, 400, 228, 40, 400, 0, 461, 35, 32],    # 6: Düsseldorf
    [190, 430, 395, 393, 471, 480, 461, 0, 400, 435], # 7: Leipzig
    [495, 588, 344, 218, 72, 412, 35, 400, 0, 34],    # 8: Dortmund
    [532, 630, 365, 253, 35, 432, 32, 435, 34, 0]     # 9: Essen
]

num_vehicles = 1
depot = 0 # Starting point: Berlin

### 2. Initializing the Routing Model & Output
Using a heuristic approach (Path Cheapest Arc) to generate an optimal decision matrix and escape local minima.

In [7]:
# Create the routing index manager and model
manager = pywrapcp.RoutingIndexManager(len(distance_matrix), num_vehicles, depot)
routing = pywrapcp.RoutingModel(manager)

# Create the distance callback function
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Set the search parameters
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

# Solve the problem
solution = routing.SolveWithParameters(search_parameters)

# Output the results neatly
if solution:
    print(f"✅ Route successfully optimized!")
    print(f"Total Objective Distance: {solution.ObjectiveValue()} kilometers\n")
    
    index = routing.Start(0)
    plan_output = 'Optimal Routing Sequence:\n'
    cities = ["Berlin", "Munich", "Hamburg", "Frankfurt", "Cologne", 
              "Stuttgart", "Düsseldorf", "Leipzig", "Dortmund", "Essen"]
              
    while not routing.IsEnd(index):
        plan_output += f' {cities[manager.IndexToNode(index)]} ->'
        index = solution.Value(routing.NextVar(index))
        
    plan_output += f' {cities[manager.IndexToNode(index)]}\n'
    print(plan_output)
else:
    print("No solution found.")

✅ Route successfully optimized!
Total Objective Distance: 1983 kilometers

Optimal Routing Sequence:
 Berlin -> Leipzig -> Munich -> Stuttgart -> Frankfurt -> Cologne -> Essen -> Düsseldorf -> Dortmund -> Hamburg -> Berlin

